# 🏥 BRECHAS EN SALUD PREVENTIVA COLOMBIA
## Módulo 2 — Modelos Predictivos v4 (Definitivo)

**Umbrales a cumplir:**

| Tarea | Variable objetivo | Umbral |
|---|---|---|
| Regresión | `TASA_MORTALIDAD_EVITABLE_100K` | RMSE ≤ 15 · MAE ≤ 10 · **R² ≥ 0.80** |
| Clasificación | `FLG_ZONA_ALTO_RIESGO` | **AUC ≥ 0.90** · **F1 ≥ 0.85** · **Recall ≥ 0.88** |

**Estrategias implementadas:**
- **S1** Lags temporales del target por municipio (mayor ganancia esperada en R²)
- **S2** Target encoding geográfico suavizado bayesiano (municipio + departamento)
- **S3** Stacking OOF: XGBoost + LightGBM + HistGB + RF → meta-Ridge / meta-LogReg
- **S4** Tuning profundo (n_iter=40) + early stopping real sobre val split 15%
- **Split** temporal out-of-time: train ≤ 2019 / test > 2019

In [ ]:
# ============================================================
# 0. IMPORTS Y CONFIGURACIÓN GLOBAL
# ============================================================
import os, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.base import clone

from sklearn.pipeline        import Pipeline
from sklearn.impute          import SimpleImputer
from sklearn.preprocessing   import RobustScaler
from sklearn.compose         import ColumnTransformer
from sklearn.model_selection import (
    KFold, StratifiedKFold,
    cross_val_score, RandomizedSearchCV, train_test_split
)
from sklearn.linear_model  import Ridge, LogisticRegression
from sklearn.ensemble      import (
    RandomForestRegressor, RandomForestClassifier,
    HistGradientBoostingRegressor, HistGradientBoostingClassifier
)
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    roc_auc_score, f1_score, recall_score, precision_score,
    confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, RocCurveDisplay
)
import xgboost  as xgb
import lightgbm as lgb
import shap
import joblib

# Semillas
SEED = 42
random.seed(SEED); np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
try:
    import torch
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'  PyTorch {torch.__version__} | device={DEVICE}')
except ImportError:
    DEVICE = 'cpu'

# Constantes
TARGET_REG  = 'TASA_MORTALIDAD_EVITABLE_100K'
TARGET_CLF  = 'FLG_ZONA_ALTO_RIESGO'
ANIO_CORTE  = 2019
UMBRAL_RMSE = 15.0
UMBRAL_MAE  = 10.0
UMBRAL_R2   = 0.80
UMBRAL_AUC  = 0.90
UMBRAL_F1   = 0.85
UMBRAL_REC  = 0.88

BASE_ML    = '/kaggle/input/datasets/nicolasacostaa/ml-salud-colombia'
OUTPUT_DIR = '/kaggle/working'

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.4f}'.format)
sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})
print('Config lista | SEED=42 | UMBRAL_R2=0.80 | UMBRAL_AUC=0.90')

In [ ]:
# ============================================================
# 1. CARGA DE DATOS
# ============================================================
df_train = pd.read_parquet(f'{BASE_ML}/tabla_hechos_train.parquet')
df_test  = pd.read_parquet(f'{BASE_ML}/tabla_hechos_test.parquet')

print(f'Train: {df_train.shape} | años {int(df_train.ANIO.min())}–{int(df_train.ANIO.max())}')
print(f'Test : {df_test.shape}  | años {int(df_test.ANIO.min())}–{int(df_test.ANIO.max())}')
print(f'\nTarget regresión (train):')
print(df_train[TARGET_REG].describe(percentiles=[.25,.5,.75,.8,.9,.95]).round(2))
print(f'\nBalance clasificación (train):')
print(df_train[TARGET_CLF].value_counts(normalize=True).round(3))

## 2. Feature Engineering

Tres bloques de features nuevas sobre la tabla de hechos base:
- **Lags temporales** del target por municipio (LAG1, LAG2, rolling mean/std, trend)
- **Target encoding** geográfico suavizado bayesiano para municipio y departamento
- **Features derivadas** espacio-temporales (interacciones año × coordenadas)

In [ ]:
# ============================================================
# 2. FEATURE ENGINEERING
# ============================================================

# ── 2.1 Lags temporales del target (anti-leakage) ─────────
def agregar_lags(df_tr, df_te, target, col_mpio='COD_MPIO', col_anio='ANIO'):
    """
    Calcula lags sobre train+test combinados ordenados por (municipio, año).
    shift(1) garantiza que nunca se usa información del año actual ni futura.
    NaN del primer año de cada municipio → imputados con mediana global de train.
    """
    df_all = pd.concat([df_tr, df_te], ignore_index=True)
    df_all = df_all.sort_values([col_mpio, col_anio]).reset_index(drop=True)
    g = df_all.groupby(col_mpio)[target]
    df_all['LAG1_TARGET']       = g.shift(1)
    df_all['LAG2_TARGET']       = g.shift(2)
    df_all['ROLL3_MEAN_TARGET'] = g.transform(
        lambda x: x.shift(1).rolling(3, min_periods=1).mean())
    df_all['ROLL3_STD_TARGET']  = g.transform(
        lambda x: x.shift(1).rolling(3, min_periods=2).std().fillna(0))
    df_all['TREND_TARGET']      = df_all['LAG1_TARGET'] - df_all['LAG2_TARGET']
    lag_cols = ['LAG1_TARGET','LAG2_TARGET','ROLL3_MEAN_TARGET',
                'ROLL3_STD_TARGET','TREND_TARGET']
    for col in lag_cols:
        med = df_all.loc[df_all[col_anio] <= ANIO_CORTE, col].median()
        df_all[col] = df_all[col].fillna(med)
    mask_tr = df_all[col_anio] <= ANIO_CORTE
    return df_all[mask_tr].reset_index(drop=True), df_all[~mask_tr].reset_index(drop=True), lag_cols


# ── 2.2 Target encoding geográfico suavizado ──────────────
def target_encoding(df_tr, df_te, col_cat, target, k=10):
    """
    Encoding bayesiano: mu_enc = (n*mu_cat + k*mu_global) / (n+k)
    Calculado solo sobre train → sin leakage.
    Municipios/deptos no vistos reciben mu_global.
    """
    mu_global  = df_tr[target].mean()
    stats_cat  = df_tr.groupby(col_cat)[target].agg(['mean','count'])
    stats_cat['encoded'] = (
        (stats_cat['count'] * stats_cat['mean'] + k * mu_global)
        / (stats_cat['count'] + k)
    )
    col_name = f'TE_{col_cat}'
    df_tr = df_tr.copy(); df_te = df_te.copy()
    df_tr[col_name] = df_tr[col_cat].map(stats_cat['encoded']).fillna(mu_global)
    df_te[col_name] = df_te[col_cat].map(stats_cat['encoded']).fillna(mu_global)
    return df_tr, df_te, col_name


# ── 2.3 Features derivadas espacio-temporales ─────────────
def agregar_derivadas(df):
    d = df.copy()
    anio_norm = (df['ANIO'] - 1979) / (2024 - 1979)
    d['ANIO_NORM']     = anio_norm
    d['LATITUD_ANIO']  = df['LATITUD']  * anio_norm
    d['LONGITUD_ANIO'] = df['LONGITUD'] * anio_norm
    d['ANIO_SQ_NORM']  = anio_norm ** 2
    d['LAT_LON']       = df['LATITUD'] * df['LONGITUD']
    return d


# ── Aplicar pipeline de FE ────────────────────────────────
print('Aplicando feature engineering...')
df_tr_fe, df_te_fe, LAG_COLS = agregar_lags(df_train, df_test, TARGET_REG)
df_tr_fe, df_te_fe, TE_MPIO  = target_encoding(df_tr_fe, df_te_fe, 'COD_MPIO', TARGET_REG, k=10)
df_tr_fe, df_te_fe, TE_DPTO  = target_encoding(df_tr_fe, df_te_fe, 'COD_DPTO', TARGET_REG, k=5)
df_tr_fe, df_te_fe, TE_MPIO_CLF = target_encoding(df_tr_fe, df_te_fe, 'COD_MPIO', TARGET_CLF, k=10)
df_tr_fe = agregar_derivadas(df_tr_fe)
df_te_fe = agregar_derivadas(df_te_fe)
TE_COLS   = [TE_MPIO, TE_DPTO]
DERIV_COLS = ['ANIO_NORM','LATITUD_ANIO','LONGITUD_ANIO','ANIO_SQ_NORM','LAT_LON']

print(f'FE completo | train: {df_tr_fe.shape} | test: {df_te_fe.shape}')
print('\nCorrelación Spearman nuevas features vs target (train):')
for c in LAG_COLS + TE_COLS:
    if c in df_tr_fe.columns:
        r, _ = stats.spearmanr(
            df_tr_fe[c].dropna(),
            df_tr_fe[TARGET_REG][df_tr_fe[c].notna()])
        print(f'  {c:30s}: {r:+.4f}')

In [ ]:
# ============================================================
# 3. DEFINICIÓN DE FEATURES
# ============================================================

# Sin features colineales perfectas (eliminadas por diseño)
FEATS_GEO      = ['LONGITUD', 'LATITUD']
FEATS_DEMO     = ['ANIO', 'POBLACION_TOTAL']
FEATS_RIPS     = ['TASA_ATENCIONES_RIPS_100K',
                  'RIPS_CONSULTAS', 'RIPS_HOSPITALIZACIONES',
                  'RIPS_PROCEDIMIENTOS_DE_SALUD', 'RIPS_URGENCIAS',
                  'PCT_CONSULTAS_100K', 'PCT_HOSPITALIZACIONES_100K',
                  'PCT_PROCEDIMIENTOS_DE_SALUD_100K', 'PCT_URGENCIAS_100K']
FEATS_GIROS    = ['GIRO_PER_CAPITA', 'N_GIROS', 'N_GIROS_ATIPICOS']
FEATS_SIVIGILA = ['TOTAL_EVENTOS_SIVIGILA', 'N_TIPOS_EVENTO', 'TASA_SIVIGILA_100K']
FEATS_BDUA     = ['BDUA_TOTAL_AFILIADOS', 'BDUA_N_ENTIDADES',
                  'PCT_AFILIACION', 'BDUA_REG_SUBSIDIADO']

ALL_FEATS = (
    FEATS_GEO + FEATS_DEMO + FEATS_RIPS + FEATS_GIROS +
    FEATS_SIVIGILA + FEATS_BDUA + DERIV_COLS + LAG_COLS + TE_COLS
)
ALL_FEATS = [f for f in ALL_FEATS if f in df_tr_fe.columns]

# Clasificación: añadir TE del target binario
ALL_FEATS_CLF = ALL_FEATS + (
    [TE_MPIO_CLF] if TE_MPIO_CLF in df_tr_fe.columns else []
)

print(f'Features regresión   : {len(ALL_FEATS)}')
print(f'Features clasificación: {len(ALL_FEATS_CLF)}')

nul = df_tr_fe[ALL_FEATS].isna().mean().sort_values(ascending=False)
print('\nTop 10 features con más nulos (train):')
print(nul.head(10).map(lambda x: f'{x:.1%}'))

In [ ]:
# ============================================================
# 4. PREPARACIÓN X / y
# ============================================================

def prep(df, feats, target):
    d = df[feats + [target]].dropna(subset=[target]).copy()
    return d[feats], d[target]

X_tr_r, y_tr_r = prep(df_tr_fe, ALL_FEATS,     TARGET_REG)
X_te_r, y_te_r = prep(df_te_fe, ALL_FEATS,     TARGET_REG)
X_tr_c, y_tr_c = prep(df_tr_fe, ALL_FEATS_CLF, TARGET_CLF)
X_te_c, y_te_c = prep(df_te_fe, ALL_FEATS_CLF, TARGET_CLF)

# Imputer base para modelos que no manejan NaN nativamente
imp_r = SimpleImputer(strategy='median').fit(X_tr_r)
imp_c = SimpleImputer(strategy='median').fit(X_tr_c)
X_tr_r_imp = imp_r.transform(X_tr_r)
X_te_r_imp = imp_r.transform(X_te_r)
X_tr_c_imp = imp_c.transform(X_tr_c)
X_te_c_imp = imp_c.transform(X_te_c)

# Sub-split 85/15 para early stopping
X_tr_sub,   X_val_sub,   y_tr_sub,   y_val_sub   = train_test_split(
    X_tr_r_imp, y_tr_r, test_size=0.15, random_state=SEED)
X_tr_sub_c, X_val_sub_c, y_tr_sub_c, y_val_sub_c = train_test_split(
    X_tr_c_imp, y_tr_c, test_size=0.15, random_state=SEED, stratify=y_tr_c)

print(f'Train reg : {X_tr_r.shape} | Test: {X_te_r.shape}')
print(f'Train clf : {X_tr_c.shape} | Test: {X_te_c.shape}')
print(f'Balance test clf: {y_te_c.value_counts(normalize=True).round(3).to_dict()}')

In [ ]:
# ============================================================
# 5. FUNCIONES DE EVALUACIÓN
# ============================================================

def eval_reg(y_true, y_pred, nombre):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + 1e-6))) * 100
    ok_r  = 'OK' if rmse <= UMBRAL_RMSE else 'X'
    ok_m  = 'OK' if mae  <= UMBRAL_MAE  else 'X'
    ok_r2 = 'OK' if r2   >= UMBRAL_R2   else 'X'
    print(f'  [{nombre:22s}] RMSE={rmse:6.3f}[{ok_r}] | MAE={mae:6.3f}[{ok_m}] | R2={r2:.4f}[{ok_r2}] | MAPE={mape:.1f}%')
    return {'modelo': nombre, 'RMSE': rmse, 'MAE': mae, 'R2': r2,
            'MAPE_%': mape, 'OK_RMSE': ok_r, 'OK_MAE': ok_m, 'OK_R2': ok_r2}


def buscar_umbral(y_true, y_proba):
    """Umbral óptimo por índice de Youden J = Sens + Espec - 1, clipeado [0.20, 0.50]."""
    fpr, tpr, thrs = roc_curve(y_true, y_proba)
    j   = tpr - fpr
    idx = np.argmax(j)
    return float(np.clip(thrs[idx], 0.20, 0.50))


def eval_clf(y_true, y_proba, nombre, umbral=None):
    if umbral is None:
        umbral = buscar_umbral(y_true, y_proba)
    y_bin = (y_proba >= umbral).astype(int)
    auc   = roc_auc_score(y_true, y_proba)
    f1    = f1_score(y_true, y_bin)
    rec   = recall_score(y_true, y_bin)
    prec  = precision_score(y_true, y_bin, zero_division=0)
    ok_a  = 'OK' if auc  >= UMBRAL_AUC else 'X'
    ok_f  = 'OK' if f1   >= UMBRAL_F1  else 'X'
    ok_r  = 'OK' if rec  >= UMBRAL_REC else 'X'
    print(f'  [{nombre:22s}] AUC={auc:.4f}[{ok_a}] | F1={f1:.4f}[{ok_f}] | Recall={rec:.4f}[{ok_r}] | Prec={prec:.4f} | umb={umbral:.3f}')
    return {'modelo': nombre, 'AUC': auc, 'F1': f1, 'Recall': rec,
            'Precision': prec, 'umbral': umbral, 'y_bin': y_bin,
            'OK_AUC': ok_a, 'OK_F1': ok_f, 'OK_Recall': ok_r}

print('Funciones de evaluación listas')

## 6. Modelos de Regresión

Cuatro modelos base + stacking OOF. Todos usan el conjunto de features v4 (base + lags + target encoding + derivadas espacio-temporales).

In [ ]:
# ============================================================
# 6.1 XGBoost Regresión — tuning profundo + early stopping
# ============================================================
print('=== XGBoost Regresion ===')

xgb_device = 'cuda' if DEVICE == 'cuda' else 'cpu'

param_grid_xgb_r = {
    'n_estimators'     : [800, 1000, 1200],
    'learning_rate'    : [0.01, 0.02, 0.05],
    'max_depth'        : [4, 5, 6],
    'min_child_weight' : [3, 5, 10],
    'subsample'        : [0.7, 0.8, 0.9],
    'colsample_bytree' : [0.6, 0.7, 0.8],
    'reg_alpha'        : [0.0, 0.1, 0.5],
    'reg_lambda'       : [1.0, 2.0, 5.0],
    'gamma'            : [0.0, 0.1, 0.3],
}

xgb_search_r = RandomizedSearchCV(
    xgb.XGBRegressor(
        tree_method='hist', device=xgb_device,
        eval_metric='rmse', verbosity=0, random_state=SEED
    ),
    param_grid_xgb_r, n_iter=40,
    cv=KFold(n_splits=5, shuffle=True, random_state=SEED),
    scoring='neg_root_mean_squared_error',
    refit=True, random_state=SEED, n_jobs=-1, verbose=0
)
xgb_search_r.fit(X_tr_r_imp, y_tr_r)
best_xgb_r = {k: v for k, v in xgb_search_r.best_params_.items() if k != 'n_estimators'}
print(f'  Mejores HPs: {best_xgb_r}')
print(f'  CV RMSE: {-xgb_search_r.best_score_:.4f}')

# Early stopping para encontrar n_estimators optimo
xgb_es_r = xgb.XGBRegressor(
    **best_xgb_r, n_estimators=3000,
    tree_method='hist', device=xgb_device,
    eval_metric='rmse', verbosity=0,
    random_state=SEED, early_stopping_rounds=50
)
xgb_es_r.fit(X_tr_sub, y_tr_sub,
             eval_set=[(X_val_sub, y_val_sub)], verbose=False)
N_XGB_R = xgb_es_r.best_iteration + 1
print(f'  Early stopping -> iteracion optima: {N_XGB_R}')

# Refit final sobre TODO el train
XGB_R = xgb.XGBRegressor(
    **best_xgb_r, n_estimators=N_XGB_R,
    tree_method='hist', device=xgb_device,
    eval_metric='rmse', verbosity=0, random_state=SEED
)
XGB_R.fit(X_tr_r_imp, y_tr_r)
y_pred_xgb_r = XGB_R.predict(X_te_r_imp)
met_xgb_r    = eval_reg(y_te_r, y_pred_xgb_r, 'XGBoost_R')

In [ ]:
# ============================================================
# 6.2 LightGBM Regresion — tuning profundo + early stopping
# ============================================================
print('=== LightGBM Regresion ===')

param_grid_lgb_r = {
    'n_estimators'      : [800, 1000, 1200],
    'learning_rate'     : [0.01, 0.02, 0.05],
    'num_leaves'        : [31, 63, 127],
    'min_child_samples' : [10, 20, 50],
    'subsample'         : [0.7, 0.8, 0.9],
    'colsample_bytree'  : [0.6, 0.7, 0.8],
    'reg_alpha'         : [0.0, 0.1, 0.5],
    'reg_lambda'        : [1.0, 2.0, 5.0],
    'min_split_gain'    : [0.0, 0.01, 0.1],
}

lgb_search_r = RandomizedSearchCV(
    lgb.LGBMRegressor(random_state=SEED, verbose=-1, n_jobs=-1),
    param_grid_lgb_r, n_iter=40,
    cv=KFold(n_splits=5, shuffle=True, random_state=SEED),
    scoring='neg_root_mean_squared_error',
    refit=True, random_state=SEED, n_jobs=-1, verbose=0
)
lgb_search_r.fit(X_tr_r_imp, y_tr_r)
best_lgb_r = {k: v for k, v in lgb_search_r.best_params_.items() if k != 'n_estimators'}
print(f'  Mejores HPs: {best_lgb_r}')
print(f'  CV RMSE: {-lgb_search_r.best_score_:.4f}')

lgb_es_r = lgb.LGBMRegressor(
    **best_lgb_r, n_estimators=3000,
    random_state=SEED, verbose=-1, n_jobs=-1
)
lgb_es_r.fit(X_tr_sub, y_tr_sub,
             eval_set=[(X_val_sub, y_val_sub)],
             callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
N_LGB_R = lgb_es_r.best_iteration_
print(f'  Early stopping -> iteracion optima: {N_LGB_R}')

LGB_R = lgb.LGBMRegressor(
    **best_lgb_r, n_estimators=N_LGB_R,
    random_state=SEED, verbose=-1, n_jobs=-1
)
LGB_R.fit(X_tr_r_imp, y_tr_r)
y_pred_lgb_r = LGB_R.predict(X_te_r_imp)
met_lgb_r    = eval_reg(y_te_r, y_pred_lgb_r, 'LightGBM_R')

In [ ]:
# ============================================================
# 6.3 HistGradientBoosting + RandomForest Regresion
# ============================================================
print('=== HistGradientBoosting Regresion ===')

hgb_search_r = RandomizedSearchCV(
    HistGradientBoostingRegressor(
        random_state=SEED, early_stopping=True,
        validation_fraction=0.15, n_iter_no_change=30
    ),
    {
        'max_leaf_nodes'   : [63, 127, 255],
        'min_samples_leaf' : [10, 20, 30],
        'learning_rate'    : [0.01, 0.02, 0.05],
        'l2_regularization': [0.0, 0.1, 1.0, 10.0],
        'max_iter'         : [500, 800],
    },
    n_iter=30,
    cv=KFold(n_splits=5, shuffle=True, random_state=SEED),
    scoring='neg_root_mean_squared_error',
    refit=True, random_state=SEED, n_jobs=-1, verbose=0
)
hgb_search_r.fit(X_tr_r, y_tr_r)  # HGB maneja NaN nativamente
HGB_R = hgb_search_r.best_estimator_
y_pred_hgb_r = HGB_R.predict(X_te_r)
met_hgb_r    = eval_reg(y_te_r, y_pred_hgb_r, 'HistGB_R')
print(f'  HPs: {hgb_search_r.best_params_}')

print('\n=== RandomForest Regresion ===')
rf_search_r = RandomizedSearchCV(
    RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1),
    {'max_depth': [10, 20, None], 'min_samples_leaf': [1, 3, 5, 10],
     'max_features': ['sqrt', 'log2', 0.5]},
    n_iter=20,
    cv=KFold(n_splits=5, shuffle=True, random_state=SEED),
    scoring='neg_root_mean_squared_error',
    refit=True, random_state=SEED, n_jobs=-1, verbose=0
)
rf_search_r.fit(X_tr_r_imp, y_tr_r)
RF_R = rf_search_r.best_estimator_
y_pred_rf_r = RF_R.predict(X_te_r_imp)
met_rf_r    = eval_reg(y_te_r, y_pred_rf_r, 'RandomForest_R')
print(f'  HPs: {rf_search_r.best_params_}')

In [ ]:
# ============================================================
# 6.4 Stacking Regresion (OOF -> meta-Ridge)
# Genera predicciones Out-Of-Fold de los 4 modelos base
# y entrena un meta-modelo Ridge sobre ellas.
# ============================================================
print('=== Stacking Regresion ===')

CV_STACK = KFold(n_splits=5, shuffle=True, random_state=SEED)
BASE_R   = {'XGB': (XGB_R,'imp'), 'LGB': (LGB_R,'imp'),
            'HGB': (HGB_R,'raw'), 'RF' : (RF_R, 'imp')}

oof_r = np.zeros((len(X_tr_r_imp), len(BASE_R)))
tst_r = np.zeros((len(X_te_r_imp), len(BASE_R)))

for i, (nm, (model, mode)) in enumerate(BASE_R.items()):
    oof_fold = np.zeros(len(X_tr_r_imp))
    tst_fold = np.zeros(len(X_te_r_imp))
    for idx_tr_f, idx_val_f in CV_STACK.split(X_tr_r_imp):
        m = clone(model)
        if mode == 'raw':
            m.fit(X_tr_r.iloc[idx_tr_f], y_tr_r.iloc[idx_tr_f])
            oof_fold[idx_val_f] = m.predict(X_tr_r.iloc[idx_val_f])
            tst_fold           += m.predict(X_te_r) / CV_STACK.n_splits
        else:
            m.fit(X_tr_r_imp[idx_tr_f], y_tr_r.iloc[idx_tr_f])
            oof_fold[idx_val_f] = m.predict(X_tr_r_imp[idx_val_f])
            tst_fold           += m.predict(X_te_r_imp) / CV_STACK.n_splits
    oof_r[:, i] = oof_fold
    tst_r[:, i] = tst_fold
    print(f'  [{nm}] OOF R2={r2_score(y_tr_r, oof_fold):.4f}')

meta_r = Ridge(alpha=1.0)
meta_r.fit(oof_r, y_tr_r)
y_pred_stack_r = meta_r.predict(tst_r)
met_stack_r    = eval_reg(y_te_r, y_pred_stack_r, 'Stacking_R')
print(f'  Pesos meta: {dict(zip(BASE_R.keys(), meta_r.coef_.round(3)))}')

In [ ]:
# ============================================================
# 6.5 Comparativa Regresion + Seleccion del mejor modelo
# ============================================================
print('\n=== RESULTADOS REGRESION v4 ===')

res_r = {
    'XGBoost_R'    : (y_pred_xgb_r,   met_xgb_r),
    'LightGBM_R'   : (y_pred_lgb_r,   met_lgb_r),
    'HistGB_R'     : (y_pred_hgb_r,   met_hgb_r),
    'RandomForest_R': (y_pred_rf_r,   met_rf_r),
    'Stacking_R'   : (y_pred_stack_r, met_stack_r),
}
df_reg = pd.DataFrame([v for _, v in res_r.values()]).set_index('modelo')
print(df_reg[['RMSE','MAE','R2','MAPE_%','OK_RMSE','OK_MAE','OK_R2']]
      .sort_values('R2', ascending=False).to_string())

cumplen_r = df_reg[(df_reg['RMSE'] <= UMBRAL_RMSE) & (df_reg['R2'] >= UMBRAL_R2)]
if len(cumplen_r) > 0:
    MEJOR_REG = cumplen_r['RMSE'].idxmin()
    print(f'\n{len(cumplen_r)} modelo(s) cumplen umbrales -> MEJOR: {MEJOR_REG}')
else:
    MEJOR_REG = df_reg['R2'].idxmax()
    gap = UMBRAL_R2 - df_reg.loc[MEJOR_REG, 'R2']
    print(f'\nNinguno cumple todos los umbrales -> mas cercano: {MEJOR_REG} (gap R2: {gap:.4f})')

y_pred_mejor_r = res_r[MEJOR_REG][0]

# Visualizacion
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()
for i, (nombre, (y_pred, _)) in enumerate(res_r.items()):
    ax = axes[i]
    ax.scatter(y_te_r, y_pred, alpha=0.2, s=6, color='#2980B9')
    lim = [min(y_te_r.min(), y_pred.min()), max(y_te_r.max(), y_pred.max())]
    ax.plot(lim, lim, 'r--', lw=1.5)
    r2_v = r2_score(y_te_r, y_pred)
    rmse_v = np.sqrt(mean_squared_error(y_te_r, y_pred))
    ok = ' [OK]' if r2_v >= UMBRAL_R2 else ''
    ax.set_title(f'{nombre}{ok}\nR2={r2_v:.4f} | RMSE={rmse_v:.2f}', fontsize=9)
    ax.set_xlabel('Real'); ax.set_ylabel('Predicho')
ax = axes[-1]
colores = ['#27AE60' if r >= UMBRAL_R2 else '#E74C3C' for r in df_reg['R2']]
df_reg['R2'].sort_values().plot.barh(ax=ax, color=colores[::-1], alpha=0.85)
ax.axvline(UMBRAL_R2, color='red', linestyle='--', label=f'Umbral={UMBRAL_R2}')
ax.set_title('R2 comparativo'); ax.legend(fontsize=8)
plt.suptitle('Regresion v4 — Real vs Predicho (test out-of-time)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/reg_v4_comparativa.png', dpi=110, bbox_inches='tight')
plt.show()

# Residuos mejor modelo
res_vals = y_te_r.values - y_pred_mejor_r
fig2, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.scatter(y_pred_mejor_r, res_vals, alpha=0.2, s=6, color='#E67E22')
ax1.axhline(0, color='red', lw=1)
ax1.set_title(f'Residuos — {MEJOR_REG}'); ax1.set_xlabel('Predicho'); ax1.set_ylabel('Residuo')
ax2.hist(res_vals, bins=60, color='#2980B9', alpha=0.8, edgecolor='white')
ax2.set_title('Distribucion residuos')
print(f'  Residuo medio: {res_vals.mean():.3f} | std: {res_vals.std():.3f}')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/reg_v4_residuos.png', dpi=110, bbox_inches='tight')
plt.show()

## 7. Modelos de Clasificación

Mismo conjunto de modelos que regresión adaptados para clasificación binaria. Se usa `class_weight='balanced'` / `scale_pos_weight` para manejar el desbalance 80/20. El umbral de clasificación se elige por el índice de Youden sobre el conjunto de entrenamiento.

In [ ]:
# ============================================================
# 7.1 XGBoost Clasificacion
# ============================================================
print('=== XGBoost Clasificacion ===')

n_neg = int((y_tr_c == 0).sum())
n_pos = int((y_tr_c == 1).sum())
spw   = n_neg / n_pos
print(f'  Desbalance: neg={n_neg} | pos={n_pos} | scale_pos_weight={spw:.2f}')

param_grid_xgb_c = {
    'n_estimators'     : [500, 800, 1000],
    'learning_rate'    : [0.01, 0.02, 0.05],
    'max_depth'        : [4, 5, 6],
    'min_child_weight' : [3, 5, 10],
    'subsample'        : [0.7, 0.8, 0.9],
    'colsample_bytree' : [0.6, 0.7, 0.8],
    'reg_alpha'        : [0.0, 0.1, 0.5],
    'reg_lambda'       : [1.0, 2.0, 5.0],
    'gamma'            : [0.0, 0.1, 0.3],
}

xgb_search_c = RandomizedSearchCV(
    xgb.XGBClassifier(
        scale_pos_weight=spw, tree_method='hist', device=xgb_device,
        eval_metric='auc', verbosity=0, random_state=SEED
    ),
    param_grid_xgb_c, n_iter=40,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    scoring='roc_auc',
    refit=True, random_state=SEED, n_jobs=-1, verbose=0
)
xgb_search_c.fit(X_tr_c_imp, y_tr_c)
best_xgb_c = {k: v for k, v in xgb_search_c.best_params_.items() if k != 'n_estimators'}
print(f'  Mejores HPs: {best_xgb_c}')
print(f'  CV AUC: {xgb_search_c.best_score_:.4f}')

xgb_es_c = xgb.XGBClassifier(
    **best_xgb_c, scale_pos_weight=spw, n_estimators=3000,
    tree_method='hist', device=xgb_device, eval_metric='auc',
    verbosity=0, random_state=SEED, early_stopping_rounds=50
)
xgb_es_c.fit(X_tr_sub_c, y_tr_sub_c,
             eval_set=[(X_val_sub_c, y_val_sub_c)], verbose=False)
N_XGB_C = xgb_es_c.best_iteration + 1
print(f'  Early stopping -> iteracion optima: {N_XGB_C}')

XGB_C = xgb.XGBClassifier(
    **best_xgb_c, scale_pos_weight=spw, n_estimators=N_XGB_C,
    tree_method='hist', device=xgb_device, eval_metric='auc',
    verbosity=0, random_state=SEED
)
XGB_C.fit(X_tr_c_imp, y_tr_c)
proba_xgb_c = XGB_C.predict_proba(X_te_c_imp)[:, 1]
umb_xgb_c   = buscar_umbral(y_tr_c, XGB_C.predict_proba(X_tr_c_imp)[:, 1])
met_xgb_c   = eval_clf(y_te_c, proba_xgb_c, 'XGBoost_C', umb_xgb_c)

In [ ]:
# ============================================================
# 7.2 LightGBM + HistGB + RandomForest Clasificacion
# ============================================================
print('=== LightGBM Clasificacion ===')

lgb_search_c = RandomizedSearchCV(
    lgb.LGBMClassifier(is_unbalance=True, random_state=SEED, verbose=-1, n_jobs=-1),
    {'n_estimators': [500, 800, 1000], 'learning_rate': [0.01, 0.02, 0.05],
     'num_leaves': [31, 63, 127], 'min_child_samples': [10, 20, 50],
     'subsample': [0.7, 0.8, 0.9], 'colsample_bytree': [0.6, 0.7, 0.8],
     'reg_alpha': [0.0, 0.1, 0.5], 'reg_lambda': [1.0, 2.0, 5.0]},
    n_iter=40,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    scoring='roc_auc', refit=True, random_state=SEED, n_jobs=-1, verbose=0
)
lgb_search_c.fit(X_tr_c_imp, y_tr_c)
best_lgb_c = {k: v for k, v in lgb_search_c.best_params_.items() if k != 'n_estimators'}
lgb_es_c   = lgb.LGBMClassifier(**best_lgb_c, is_unbalance=True,
                                  n_estimators=3000, random_state=SEED, verbose=-1, n_jobs=-1)
lgb_es_c.fit(X_tr_sub_c, y_tr_sub_c, eval_set=[(X_val_sub_c, y_val_sub_c)],
             callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
N_LGB_C = lgb_es_c.best_iteration_
LGB_C   = lgb.LGBMClassifier(**best_lgb_c, is_unbalance=True, n_estimators=N_LGB_C,
                               random_state=SEED, verbose=-1, n_jobs=-1)
LGB_C.fit(X_tr_c_imp, y_tr_c)
proba_lgb_c = LGB_C.predict_proba(X_te_c_imp)[:, 1]
umb_lgb_c   = buscar_umbral(y_tr_c, LGB_C.predict_proba(X_tr_c_imp)[:, 1])
met_lgb_c   = eval_clf(y_te_c, proba_lgb_c, 'LightGBM_C', umb_lgb_c)
print(f'  HPs: {best_lgb_c} | iter={N_LGB_C}')

print('\n=== HistGB Clasificacion ===')
hgb_search_c = RandomizedSearchCV(
    HistGradientBoostingClassifier(random_state=SEED, early_stopping=True,
                                    validation_fraction=0.15, n_iter_no_change=30,
                                    class_weight='balanced'),
    {'max_leaf_nodes': [63, 127, 255], 'min_samples_leaf': [10, 20, 30],
     'learning_rate': [0.01, 0.02, 0.05], 'l2_regularization': [0.0, 0.1, 1.0],
     'max_iter': [500, 800]},
    n_iter=30,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    scoring='roc_auc', refit=True, random_state=SEED, n_jobs=-1, verbose=0
)
hgb_search_c.fit(X_tr_c, y_tr_c)
HGB_C       = hgb_search_c.best_estimator_
proba_hgb_c = HGB_C.predict_proba(X_te_c)[:, 1]
umb_hgb_c   = buscar_umbral(y_tr_c, HGB_C.predict_proba(X_tr_c)[:, 1])
met_hgb_c   = eval_clf(y_te_c, proba_hgb_c, 'HistGB_C', umb_hgb_c)

print('\n=== RandomForest Clasificacion ===')
rf_search_c = RandomizedSearchCV(
    RandomForestClassifier(n_estimators=300, class_weight='balanced',
                           random_state=SEED, n_jobs=-1),
    {'max_depth': [10, 20, None], 'min_samples_leaf': [1, 3, 5, 10],
     'max_features': ['sqrt', 'log2', 0.5]},
    n_iter=20,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    scoring='roc_auc', refit=True, random_state=SEED, n_jobs=-1, verbose=0
)
rf_search_c.fit(X_tr_c_imp, y_tr_c)
RF_C        = rf_search_c.best_estimator_
proba_rf_c  = RF_C.predict_proba(X_te_c_imp)[:, 1]
umb_rf_c    = buscar_umbral(y_tr_c, RF_C.predict_proba(X_tr_c_imp)[:, 1])
met_rf_c    = eval_clf(y_te_c, proba_rf_c, 'RandomForest_C', umb_rf_c)

In [ ]:
# ============================================================
# 7.3 Stacking Clasificacion (OOF -> meta-LogisticRegression)
# ============================================================
print('=== Stacking Clasificacion ===')

CV_STACK_C = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
BASE_C     = {'XGB': (XGB_C,'imp'), 'LGB': (LGB_C,'imp'),
              'HGB': (HGB_C,'raw'), 'RF' : (RF_C, 'imp')}

oof_c = np.zeros((len(X_tr_c_imp), len(BASE_C)))
tst_c = np.zeros((len(X_te_c_imp), len(BASE_C)))

for i, (nm, (model, mode)) in enumerate(BASE_C.items()):
    oof_fold = np.zeros(len(X_tr_c_imp))
    tst_fold = np.zeros(len(X_te_c_imp))
    for idx_tr_f, idx_val_f in CV_STACK_C.split(X_tr_c_imp, y_tr_c):
        m = clone(model)
        if mode == 'raw':
            m.fit(X_tr_c.iloc[idx_tr_f], y_tr_c.iloc[idx_tr_f])
            oof_fold[idx_val_f] = m.predict_proba(X_tr_c.iloc[idx_val_f])[:, 1]
            tst_fold           += m.predict_proba(X_te_c)[:, 1] / CV_STACK_C.n_splits
        else:
            m.fit(X_tr_c_imp[idx_tr_f], y_tr_c.iloc[idx_tr_f])
            oof_fold[idx_val_f] = m.predict_proba(X_tr_c_imp[idx_val_f])[:, 1]
            tst_fold           += m.predict_proba(X_te_c_imp)[:, 1] / CV_STACK_C.n_splits
    oof_c[:, i] = oof_fold
    tst_c[:, i] = tst_fold
    print(f'  [{nm}] OOF AUC={roc_auc_score(y_tr_c, oof_fold):.4f}')

meta_c        = LogisticRegression(C=1.0, random_state=SEED, max_iter=500)
meta_c.fit(oof_c, y_tr_c)
proba_stack_c = meta_c.predict_proba(tst_c)[:, 1]
umb_stack_c   = buscar_umbral(y_tr_c, meta_c.predict_proba(oof_c)[:, 1])
met_stack_c   = eval_clf(y_te_c, proba_stack_c, 'Stacking_C', umb_stack_c)
print(f'  Pesos meta: {dict(zip(BASE_C.keys(), meta_c.coef_[0].round(3)))}')

In [ ]:
# ============================================================
# 7.4 Comparativa Clasificacion + Seleccion del mejor modelo
# ============================================================
print('\n=== RESULTADOS CLASIFICACION v4 ===')

res_c = {
    'XGBoost_C'     : (proba_xgb_c,   umb_xgb_c,   met_xgb_c),
    'LightGBM_C'    : (proba_lgb_c,   umb_lgb_c,   met_lgb_c),
    'HistGB_C'      : (proba_hgb_c,   umb_hgb_c,   met_hgb_c),
    'RandomForest_C': (proba_rf_c,    umb_rf_c,    met_rf_c),
    'Stacking_C'    : (proba_stack_c, umb_stack_c, met_stack_c),
}
df_clf = pd.DataFrame([v for _, _, v in res_c.values()]).set_index('modelo')
print(df_clf[['AUC','F1','Recall','Precision','umbral','OK_AUC','OK_F1','OK_Recall']]
      .sort_values('AUC', ascending=False).to_string())

cumplen_c = df_clf[
    (df_clf['AUC']    >= UMBRAL_AUC) &
    (df_clf['F1']     >= UMBRAL_F1)  &
    (df_clf['Recall'] >= UMBRAL_REC)
]
if len(cumplen_c) > 0:
    MEJOR_CLF = cumplen_c['AUC'].idxmax()
    print(f'\n{len(cumplen_c)} modelos cumplen umbrales -> MEJOR: {MEJOR_CLF}')
else:
    MEJOR_CLF = df_clf['AUC'].idxmax()
    print(f'\nNinguno cumple todos -> mas cercano: {MEJOR_CLF}')

proba_mejor_c, umb_mejor_c, _ = res_c[MEJOR_CLF]
y_bin_mejor_c = (proba_mejor_c >= umb_mejor_c).astype(int)
print(f'\nReporte completo [{MEJOR_CLF}]:')
print(classification_report(y_te_c, y_bin_mejor_c, target_names=['No Riesgo','Alto Riesgo']))

# Visualizacion
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
ax = axes[0]
for nombre, (proba, umb, _) in res_c.items():
    auc_v = roc_auc_score(y_te_c, proba)
    RocCurveDisplay.from_predictions(y_te_c, proba, name=f'{nombre} ({auc_v:.3f})', ax=ax)
ax.plot([0,1],[0,1],'k--',lw=1)
ax.set_title('Curvas ROC'); ax.legend(fontsize=7)
ax2 = axes[1]
for nombre, (proba, umb, _) in res_c.items():
    prec_vals, rec_vals, _ = precision_recall_curve(y_te_c, proba)
    auc_pr = float(np.trapz(prec_vals[::-1], rec_vals[::-1]))
    ax2.plot(rec_vals, prec_vals, label=f'{nombre} (PR={auc_pr:.3f})', lw=1.5)
    y_b = (proba >= umb).astype(int)
    ax2.scatter([recall_score(y_te_c,y_b)],
                [precision_score(y_te_c,y_b,zero_division=0)], marker='*', s=150, zorder=5)
ax2.set_xlabel('Recall'); ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall (estrella=umbral optimo)'); ax2.legend(fontsize=7)
ax3 = axes[2]
cm = confusion_matrix(y_te_c, y_bin_mejor_c)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax3,
            xticklabels=['No Riesgo','Alto Riesgo'],
            yticklabels=['No Riesgo','Alto Riesgo'])
ax3.set_title(f'Confusion — {MEJOR_CLF}')
plt.suptitle('Clasificacion v4 — Test Out-of-Time', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/clf_v4_comparativa.png', dpi=110, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 8. INTERPRETABILIDAD SHAP
# ============================================================

def shap_summary(model, X_np, feat_names, titulo, fname):
    try:
        explainer   = shap.TreeExplainer(model)
        idx         = np.random.choice(len(X_np), min(500, len(X_np)), replace=False)
        shap_vals   = explainer.shap_values(X_np[idx])
        if isinstance(shap_vals, list):
            shap_vals = shap_vals[1]
        plt.figure(figsize=(10, 7))
        shap.summary_plot(shap_vals, X_np[idx], feature_names=feat_names,
                          max_display=15, show=False)
        plt.title(titulo, fontsize=11, fontweight='bold')
        plt.tight_layout()
        plt.savefig(f'{OUTPUT_DIR}/{fname}', dpi=110, bbox_inches='tight')
        plt.show()
        imp = pd.Series(np.abs(shap_vals).mean(axis=0),
                        index=feat_names).sort_values(ascending=False)
        print(f'Top 5 [{titulo}]:')
        print(imp.head(5).round(4).to_string())
    except Exception as e:
        print(f'SHAP error: {e}')

# Elegir modelo base para SHAP (XGBoost o LightGBM, siempre TreeExplainer)
model_shap_r = XGB_R if MEJOR_REG != 'Stacking_R' else XGB_R
model_shap_c = XGB_C if MEJOR_CLF != 'Stacking_C' else XGB_C

shap_summary(model_shap_r, X_te_r_imp, list(X_tr_r.columns),
             f'SHAP Regresion ({MEJOR_REG})', 'shap_reg_v4.png')
shap_summary(model_shap_c, X_te_c_imp, list(X_tr_c.columns),
             f'SHAP Clasificacion ({MEJOR_CLF})', 'shap_clf_v4.png')

In [ ]:
# ============================================================
# 9. PERSISTENCIA
# ============================================================
joblib.dump(XGB_R,  f'{OUTPUT_DIR}/xgb_reg_v4.pkl')
joblib.dump(LGB_R,  f'{OUTPUT_DIR}/lgb_reg_v4.pkl')
joblib.dump(HGB_R,  f'{OUTPUT_DIR}/hgb_reg_v4.pkl')
joblib.dump(RF_R,   f'{OUTPUT_DIR}/rf_reg_v4.pkl')
joblib.dump(meta_r, f'{OUTPUT_DIR}/meta_ridge_reg_v4.pkl')
joblib.dump(XGB_C,  f'{OUTPUT_DIR}/xgb_clf_v4.pkl')
joblib.dump(LGB_C,  f'{OUTPUT_DIR}/lgb_clf_v4.pkl')
joblib.dump(HGB_C,  f'{OUTPUT_DIR}/hgb_clf_v4.pkl')
joblib.dump(RF_C,   f'{OUTPUT_DIR}/rf_clf_v4.pkl')
joblib.dump(meta_c, f'{OUTPUT_DIR}/meta_lr_clf_v4.pkl')
joblib.dump(imp_r,  f'{OUTPUT_DIR}/imputer_reg_v4.pkl')
joblib.dump(imp_c,  f'{OUTPUT_DIR}/imputer_clf_v4.pkl')

meta_v4 = {
    'version'           : 'v4',
    'features_reg'      : list(X_tr_r.columns),
    'features_clf'      : list(X_tr_c.columns),
    'lag_cols'          : LAG_COLS,
    'te_cols_reg'       : TE_COLS,
    'te_col_clf'        : TE_MPIO_CLF,
    'mejor_reg'         : MEJOR_REG,
    'mejor_clf'         : MEJOR_CLF,
    'umbral_clf'        : float(umb_mejor_c),
    'best_params_xgb_r' : best_xgb_r,
    'best_params_lgb_r' : best_lgb_r,
    'best_params_xgb_c' : best_xgb_c,
    'best_params_lgb_c' : best_lgb_c,
    'n_opt_xgb_r'       : int(N_XGB_R),
    'n_opt_lgb_r'       : int(N_LGB_R),
    'n_opt_xgb_c'       : int(N_XGB_C),
    'n_opt_lgb_c'       : int(N_LGB_C),
    'metricas_reg'      : df_reg[['RMSE','MAE','R2','MAPE_%']].to_dict(),
    'metricas_clf'      : df_clf[['AUC','F1','Recall','Precision','umbral']].to_dict(),
}
joblib.dump(meta_v4, f'{OUTPUT_DIR}/meta_modelos_v4.pkl')
print('Artefactos v4 guardados en /kaggle/working/')

In [ ]:
# ============================================================
# 10. RESUMEN EJECUTIVO v4
# ============================================================
sep = '=' * 65
print(sep)
print('   RESUMEN EJECUTIVO — MODULO 2 v4')
print('   Brechas en Salud Preventiva Colombia')
print(sep)
print(f'''
Features regresion   : {len(list(X_tr_r.columns))}
Features clasificacion: {len(list(X_tr_c.columns))}
  Nuevas v4: lags temporales + target encoding + derivadas espacio-temporales
  Split: train <= {ANIO_CORTE} | test > {ANIO_CORTE} (out-of-time)
  Modelos: XGBoost · LightGBM · HistGB · RandomForest · Stacking OOF
''')
print('REGRESION:')
print(df_reg[['RMSE','MAE','R2','OK_RMSE','OK_MAE','OK_R2']].sort_values('R2', ascending=False).to_string())
print('\nCLASIFICACION:')
print(df_clf[['AUC','F1','Recall','Precision','umbral','OK_AUC','OK_F1','OK_Recall']].sort_values('AUC', ascending=False).to_string())
print(f'''
MEJOR REGRESION    : {MEJOR_REG}
  R2   = {df_reg.loc[MEJOR_REG,'R2']:.4f}   | Umbral >= {UMBRAL_R2}
  RMSE = {df_reg.loc[MEJOR_REG,'RMSE']:.3f} | Umbral <= {UMBRAL_RMSE}
  MAE  = {df_reg.loc[MEJOR_REG,'MAE']:.3f}  | Umbral <= {UMBRAL_MAE}

MEJOR CLASIFICACION: {MEJOR_CLF}
  AUC    = {df_clf.loc[MEJOR_CLF,'AUC']:.4f}    | Umbral >= {UMBRAL_AUC}
  F1     = {df_clf.loc[MEJOR_CLF,'F1']:.4f}    | Umbral >= {UMBRAL_F1}
  Recall = {df_clf.loc[MEJOR_CLF,'Recall']:.4f} | Umbral >= {UMBRAL_REC}
  Umbral optimo Youden J: {umb_mejor_c:.3f}
''')
print(sep)